In [3]:
import pymysql
import hashlib
import os
from cryptography.fernet import Fernet
from datetime import datetime, timedelta
import logging


In [15]:
def get_connection():
    return pymysql.connect(
        host='localhost',
        user='root',
        password='Dfyzdfyz1993!',  # ← замените!
        # database='webinar_security',
        charset='utf8mb4',
        cursorclass=pymysql.cursors.DictCursor
    )


In [19]:
conn = get_connection()
with conn.cursor() as cursor:
    # Создаём БД (если нет)
    cursor.execute("CREATE DATABASE IF NOT EXISTS webinar_security CHARACTER SET utf8mb4")
            
    # Переключаемся на неё
    cursor.execute("USE webinar_security")
    
    # Таблица с зашифрованным паролем
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS users_secure (
            id INT AUTO_INCREMENT PRIMARY KEY,
            username VARCHAR(50) NOT NULL,
            encrypted_password VARBINARY(255) NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )""")
    conn.commit()
# finally:
#     conn.close()

In [20]:
# conn = get_connection()

with conn.cursor() as cursor:
    sql = """
        INSERT INTO users_secure (username, encrypted_password)
        VALUES (%s, AES_ENCRYPT(%s, %s))
    """
    cursor.execute(sql, ('alice', 'supersecret123', 'my_secret_key'))
conn.commit()
print("Пароль зашифрован и сохранён.")
# finally:
#     conn.close()


Пароль зашифрован и сохранён.


In [26]:
# conn = get_connection()

with conn.cursor() as cursor:
    sql = "SELECT id, username, AES_DECRYPT(encrypted_password, %s) AS plaintext FROM users_secure WHERE username = %s"
    cursor.execute(sql, ('my_secret_key', 'alice'))
    result = cursor.fetchone()
    if result:
        print("ID:", result['id'])
        print("Имя:", result['username'])
        print("Пароль (расшифрованный):", result['plaintext'])
# finally:
#     conn.close()


ID: 1
Имя: alice
Пароль (расшифрованный): b'supersecret123'


In [28]:
plaintext = "секретный пароль"
plaintext.encode()

b'\xd1\x81\xd0\xb5\xd0\xba\xd1\x80\xd0\xb5\xd1\x82\xd0\xbd\xd1\x8b\xd0\xb9 \xd0\xbf\xd0\xb0\xd1\x80\xd0\xbe\xd0\xbb\xd1\x8c'

In [27]:
# Генерация ключа (один раз!)
fernet_key = Fernet.generate_key()
fernet = Fernet(fernet_key)
print("Ключ (сохраните!):", fernet_key.decode())

# Шифрование
plaintext = "секретный пароль"
ciphertext = fernet.encrypt(plaintext.encode())
print("Зашифровано:", ciphertext)


# Расшифровка
decrypted = fernet.decrypt(ciphertext).decode()
print("Расшифровано:", decrypted)


Ключ (сохраните!): aOoaD7cmyFv0SLawgIv4F9f4tZcj043Xu4mmC49864M=
Зашифровано: b'gAAAAABpKHx4OdKiQAanLIsqaH21jrRtx9od6JpyalN2rqsZfzAD5W2xWzJiq9CQ0oBVBKcu2UxPu89NN4FLsZGFHFaBXifYA9m4HjLciNvucWLETVrHFLc='
Расшифровано: секретный пароль


In [31]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding

# Генерация ключей
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)
public_key = private_key.public_key()

# Шифрование публичным ключом
message = "секретное сообщение"
encrypted = public_key.encrypt(
    message.encode(),
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)
print("Зашифровано (RSA):", encrypted)


# Расшифровка приватным ключом
decrypted = private_key.decrypt(
    encrypted,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
).decode()
print("Расшифровано (RSA):", decrypted)


Зашифровано (RSA): b'J\x81\xa5s\xe7\x07\xbdqR\xf1\xa3pD\\x\xcd\xf7\x96\xc5\xda\x1do\x94\xe7vJ\xa1\x84,\x15Y\x95\xb6\xa4K\x16\xf2\x97]\xcb\xac\xfb\xf7EM\x01\xb8D\x91\x084!N\xe6\xc7BE}S\xec\xa2p\xec\x805\xff\x8c\x96P\xe99\xa8\xe1\xf1\x98\x87L\xda\xfdV\x88\x08\xef|\x14-U0y~w\xaf\xd5\x02\xa4ng\x95\x87\xd5A\x0b\xc4\xc1?\x9d\xefz\xee\xcfmp\xf4\x89\x1e\xca\xde\x16\x87\xd3z\xec\xfb\xb8\xae\xd4\xf9z\xcdL\x17U\x1aiQ\xcaq\x82\xc3\x10[\x17\x9b;\xac\xf2\xfc\xa3\xd2\xff\x10\x99\xa6\x97l\x91\x7fd\x8d\xd1\x95\x8e\xc0\x1d\x91\x04\xca\xb9S\x02\xe8\xc7\x02\n\xd1\xd3\x1c\x99\xd4\xf9|\xbaj\xb4}\xab \xe9G\x99\xe6UW\xf9\xcfO\xa0\x1a\x94F\xcb\x1a\xfa\x99\xf8\xe2\xf3\x13\x15\x90D\xb7\xff\xbd\xfe\xfb\xda\xc4\x05\xc6\xde%\xea\xfaE\xa1d;u\x16\x93wq[h*<\xc6\x08\xc3!\xd9\xfeW\xbc\xbe\xa2\xe1\n\x81V\xdd\x8b#e"'
Расшифровано (RSA): секретное сообщение


In [32]:
# conn = get_connection()
# try:
with conn.cursor() as cursor:
    cursor.execute("USE webinar_security")
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS password_store (
            id INT AUTO_INCREMENT PRIMARY KEY,
            login VARCHAR(50) UNIQUE NOT NULL,
            encrypted_password VARBINARY(255) NOT NULL,
            salt VARBINARY(32) NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    # finally:
    #     conn.close()


In [33]:
os.urandom(32)

b'\x17a\x12\x8e\xc2\x83\xdd\xbcp\xad\xfe\xb3\xd1\x81p\xe1\x9fB\x97>\x89\xb5\x8bS\xb1\xe3\x0bhJ\xea\xd1v'

In [34]:
def register_user(login: str, password: str):
    # Генерируем соль
    salt = os.urandom(32)
    # Хешируем пароль с солью
    pwd_hash = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
    
    # Шифруем хеш
    fernet = Fernet(fernet_key)  # используем ключ из п. 2.1
    encrypted_pwd = fernet.encrypt(pwd_hash)
    

    # conn = get_connection()
    # try:
    with conn.cursor() as cursor:
        sql = """
            INSERT INTO password_store (login, encrypted_password, salt)
            VALUES (%s, %s, %s)
        """
        cursor.execute(sql, (login, encrypted_pwd, salt))
    conn.commit()
    print(f"Пользователь {login} зарегистрирован.")
    # finally:
    #     conn.close()

register_user("alice", "mypass123")


Пользователь alice зарегистрирован.


In [38]:
salt = os.urandom(32)
password = "mypass123"
pwd_hash = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
print(pwd_hash)
pwd_hash = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
print(pwd_hash)
pwd_hash = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
print(pwd_hash)
pwd_hash = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
print(pwd_hash)

b'\xc0\xc3l\xe43Mk"\xee\xfe[\xb4\xbe;=i\xe1\xec\xb8\xf5\x11\xd1\xbe\xbfw{\xe2\x0b$\xc2wZ'
b'\xc0\xc3l\xe43Mk"\xee\xfe[\xb4\xbe;=i\xe1\xec\xb8\xf5\x11\xd1\xbe\xbfw{\xe2\x0b$\xc2wZ'
b'\xc0\xc3l\xe43Mk"\xee\xfe[\xb4\xbe;=i\xe1\xec\xb8\xf5\x11\xd1\xbe\xbfw{\xe2\x0b$\xc2wZ'
b'\xc0\xc3l\xe43Mk"\xee\xfe[\xb4\xbe;=i\xe1\xec\xb8\xf5\x11\xd1\xbe\xbfw{\xe2\x0b$\xc2wZ'


In [36]:
salt = os.urandom(32)
password = "mypass123"
pwd_hash = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
pwd_hash

b'\xc9V\x80\xf6.\x01\xd5<ZJZ\xae:#\xf8w=\xa2s\x102g@\xbdC\t\xc54\xedO@\xb5'

In [39]:
def login_user(login: str, password: str) -> bool:
    # conn = get_connection()
    # try:
    with conn.cursor() as cursor:
        sql = "SELECT encrypted_password, salt FROM password_store WHERE login = %s"
        cursor.execute(sql, (login,))
        row = cursor.fetchone()
        if not row:
            return False

        encrypted_pwd, salt = row['encrypted_password'], row['salt']
        # Расшифровываем хеш
        fernet = Fernet(fernet_key)
        try:
            stored_hash = fernet.decrypt(encrypted_pwd)
        except Exception as e:
            print("Ошибка расшифровки:", e)
            return False

        # Сравниваем хеши
        input_hash = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
        return stored_hash == input_hash
    # finally:
    #     conn.close()

print("Вход (верный пароль):", login_user("alice", "mypass123"))
print("Вход (неверный пароль):", login_user("alice", "wrongpass"))


Вход (верный пароль): True
Вход (неверный пароль): False


In [40]:
# conn = get_connection()
# try:
with conn.cursor() as cursor:
    cursor.execute("USE webinar_security")
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS access_log (
            id INT AUTO_INCREMENT PRIMARY KEY,
            login VARCHAR(50),
            success BOOLEAN,
            ip_address VARCHAR(45),
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            user_agent TEXT
        )
    """)
conn.commit()
# finally:
#     conn.close()


In [42]:
def log_access(login: str, success: bool, ip: str, user_agent: str):
    # conn = get_connection()
    # try:
    with conn.cursor() as cursor:
        sql = """
            INSERT INTO access_log (login, success, ip_address, user_agent)
            VALUES (%s, %s, %s, %s)
        """
        cursor.execute(sql, (login, success, ip, user_agent))
    conn.commit()
    logging.info(f"Login attempt: {login}, Success: {success}, IP: {ip}")
    # finally:
    #     conn.close()


In [43]:
def detect_brute_force(login: str, window_minutes: int = 15) -> bool:
    window_start = datetime.now() - timedelta(minutes=window_minutes)
    # conn = get_connection()
    # try:
    with conn.cursor() as cursor:
        sql = """
            SELECT COUNT(*) FROM access_log
            WHERE login = %s AND success = 0 AND timestamp >= %s
        """
        cursor.execute(sql, (login, window_start))
        count = cursor.fetchone()['COUNT(*)']
        return count > 5
    # finally:
    #     conn.close()


In [47]:
def send_alert(login: str, ip: str):
    msg = f"ALERT: Brute force detected for user {login} from IP {ip}"
    print(msg)  # В демо — просто вывод в консоль
    # В реальности: отправка email/Slack/в SIEM


In [48]:
def login_with_monitoring(login: str, password: str, ip: str, user_agent: str) -> bool:
    success = login_user(login, password)
    log_access(login, success, ip, user_agent)

    if not success and detect_brute_force(login):
        send_alert(login, ip)

    return success

# Тестируем
login_with_monitoring("alice", "wrongpass", "192.168.1.100", "Mozilla/5.0")
login_with_monitoring("alice", "wrongpass", "192.168.1.100", "Mozilla/5.0")
# Повторите 5+ раз — сработает alert


ALERT: Brute force detected for user alice from IP 192.168.1.100
ALERT: Brute force detected for user alice from IP 192.168.1.100


False

In [45]:
login_with_monitoring("alice", "wrongpass", "192.168.1.100", "Mozilla/5.0")
login_with_monitoring("alice", "wrongpass", "192.168.1.100", "Mozilla/5.0")
login_with_monitoring("alice", "wrongpass", "192.168.1.100", "Mozilla/5.0")
login_with_monitoring("alice", "wrongpass", "192.168.1.100", "Mozilla/5.0")

NameError: name 'send_alert' is not defined